### This notebook will contain a variant of the Rocket Richard Baseline Predictor made in rr1.ipynb, that is, this variant will be using General Skater Stats (GSS) + EDGE Stats
#### This is done for the sake of model fine tuning and evaluating performance

In [48]:
import numpy as np
import pandas as pd
from nhlpy import NHLClient
import ast
from helpersrr import clear_csv, extractPlayerID, placeToStats, fetchSkaterStats, labelWinners, formatEdgeStats
from sklearn.linear_model import LogisticRegression


import importlib
import helpersrr
importlib.reload(helpersrr)

<module 'helpersrr' from '/Users/Joaquin/Documents/GitHub/nhl-trophy-predictor/notebooks/helpersrr.py'>

In [ ]:
#global variables
ranks = True
versionA = False    #version A has no shot details, version B has ALL shot details

client = NHLClient()
rocket_richard = clear_csv("../data/formattedwebscraped/rocketrichard.csv")

first_place = rocket_richard[['szn','winner']]
first_ids = placeToStats(first_place)
second_place = rocket_richard[['szn', 'runner_up']]
second_ids = placeToStats(second_place)
third_place = rocket_richard[['szn', 'finalist']]
third_ids = placeToStats(third_place)

../data/formattedwebscraped/rocketrichard.csv


### Milestone 1: Get EDGE Statistics from the API and place them in data/api/EDGEstats

In [50]:
def fetchSkaterStats(year, csv=False, edge=False):      #optimized version of fetchSkaterStats
    if len(year) == 4:
        year_interval = f"{int(year)}{int(year)+1}"
    elif len(year) == 8:
        year_interval = year
    else:
        raise SyntaxError("requires yyyy or yyyyyyyy format of season year")

    page_size = 100
    rows = []

    if edge:
        for start in range(0, 1000, page_size):
            statChunk = client.stats.skater_stats_summary(
                start_season=year_interval,
                end_season=year_interval,
                start=start,
                limit=page_size
            )
            if not statChunk:
                break

            ids = pd.DataFrame(statChunk)['playerId'].tolist()
            for player_id in ids:
                try:
                    individual_stat = client.edge.skater_detail(
                        player_id=player_id,
                        season=year_interval
                    )
                    rows.append(formatEdgeStats(individual_stat, shotDetails=True))
                except Exception as exc:
                    print("edge fetch failed", player_id, exc)

        df = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

    else:
        for start in range(0, 1000, page_size):
            statChunk = client.stats.skater_stats_summary(
                start_season=year_interval,
                end_season=year_interval,
                start=start,
                limit=page_size
            )
            if not statChunk:
                break
            rows.extend(statChunk)

        df = pd.DataFrame(rows)

    if csv:
        out = f"../data/api/EDGEstats/skatersEDGE{year_interval}.csv" if edge else f"../data/api/skaters/skaters{year_interval}.csv"
        df.to_csv(out, index=False)
    else:
        return df

In [51]:
#DO NOT RUN THIS IF COMMENTED -- essentially places relevant EDGE stats files into the data/api/EDGEstats folder
#edgeSzns = ['20212022','20222023','20232024','20242025','20252026']
#fetchSkaterStats('20212022',csv=True,edge=True)
#for season in edgeSzns:
#    fetchSkaterStats(season, csv=True, edge=True)
#    print(f"done season {season}")

In [52]:
#this cell is where formatEdgeStats() was originally made
cm97 = client.edge.skater_detail(player_id='8478402', season='20252026')
formttedcm97 = formatEdgeStats(cm97, True)
formttedcm97


,playerId,topShotSpeed,skatingSpeed,totalDistanceSkated,distanceMaxGame,longShots,longGoals,midShots,midGoals,highShots,...,L Net Side Shots,L Point Shots,Low Slot Shots,Offensive Neutral Zone Shots,Outside L Shots,Outside R Shots,R Circle Shots,R Corner Shots,R Net Side Shots,R Point Shots
0,8478402,132.0467,39.6089,531.4875,7.908,8,1,92,9,120,...,34,3,97,3,15,9,29,0,12,1


In [53]:
#will have to fetch stats for each player in each season, but after preprocessing
df = pd.DataFrame(client.stats.skater_stats_summary(
    start_season="20252026",
    end_season="20252026"))

ids = df['playerId']
new_df = []
for id in ids:
    ind_stat = client.edge.skater_detail(player_id=id)
    formatted_stat = formatEdgeStats(ind_stat, shotDetails=True)
    new_df.append(formatted_stat)

### Milestone 2: Make the new feature set (GSS + EDGE Stats)

In [54]:
#experimenting how to merge the GSS + EDGE stats; note: this is limited to 25 players since the limit exists
final_df = pd.DataFrame()
for item in new_df:
    final_df = pd.concat([final_df,item])                           #combine all EDGE stat records
regularDf = fetchSkaterStats("20252026", csv=False, edge=False)     #then combine it with regular GSS
regularDf = regularDf.merge(final_df)

with pd.option_context("display.max_rows", None, "display.max_columns", None):     #view whole dataset
        display(regularDf)

,assists,evGoals,evPoints,faceoffWinPct,gameWinningGoals,gamesPlayed,goals,lastName,otGoals,penaltyMinutes,playerId,plusMinus,points,pointsPerGame,positionCode,ppGoals,ppPoints,seasonId,shGoals,shPoints,shootingPct,shootsCatches,shots,skaterFullName,teamAbbrevs,timeOnIcePerGame,topShotSpeed,skatingSpeed,totalDistanceSkated,distanceMaxGame,longShots,longGoals,midShots,midGoals,highShots,highGoals,offensiveZonePctg,neutralZonePctg,defensiveZonePctg,Behind the Net Shots,Beyond Red Line Shots,Center Point Shots,Crease Shots,High Slot Shots,L Circle Shots,L Corner Shots,L Net Side Shots,L Point Shots,Low Slot Shots,Offensive Neutral Zone Shots,Outside L Shots,Outside R Shots,R Circle Shots,R Corner Shots,R Net Side Shots,R Point Shots
0,90,34,82,0.49512,4,82,48,McDavid,1,44,8478402,17,138,1.68292,C,13,54,20252026,1,2,0.15686,L,306,Connor McDavid,EDM,1379.1219,132.0467,39.6089,531.4875,7.9080,8,1,92,9,120,26,0.476879,0.169191,0.353930,11,1,4,23,27,36,1,34,3,97,3,15,9,29,0,12,1
1,86,35,92,0.00000,8,76,44,Kucherov,2,50,8476453,43,130,1.71052,R,8,37,20252026,1,1,0.19047,L,231,Nikita Kucherov,TBL,1220.1052,148.1562,36.1133,392.7248,7.2929,38,3,78,12,45,18,0.456801,0.185946,0.357253,2,1,7,5,25,16,2,6,9,40,2,6,40,37,0,11,22
2,74,42,97,0.51049,7,80,53,MacKinnon,1,39,8477492,57,127,1.58750,C,11,30,20252026,0,0,0.15142,R,350,Nathan MacKinnon,COL,1335.7125,146.4342,39.8786,456.0059,7.8713,32,6,149,20,83,19,0.466145,0.180970,0.352885,3,3,16,11,42,77,1,19,11,72,5,38,12,30,0,5,5
3,70,37,82,0.48638,5,82,45,Celebrini,2,44,8484801,8,115,1.40243,C,8,33,20252026,0,0,0.15679,L,287,Macklin Celebrini,SJS,1278.9512,157.5387,35.9208,472.6669,7.5709,28,3,159,25,59,13,0.445342,0.181240,0.373417,2,2,20,5,66,52,0,4,3,54,5,17,6,41,0,5,5
4,67,29,80,0.46661,6,82,36,Scheifele,1,43,8476460,0,103,1.25609,C,7,22,20252026,0,1,0.20571,R,175,Mark Scheifele,WPG,1290.2439,139.8681,37.7198,467.6965,7.1238,4,1,70,18,66,11,0.439736,0.176693,0.383570,3,1,3,6,20,31,1,11,0,60,2,7,8,19,1,1,1
5,72,17,57,0.50448,2,82,29,Suzuki,1,28,8480018,37,101,1.23170,C,11,43,20252026,1,1,0.15846,R,183,Nick Suzuki,MTL,1249.2317,138.4358,38.0149,437.0927,6.7465,11,0,66,9,52,10,0.451094,0.173137,0.375769,2,3,5,6,20,29,1,8,3,46,3,18,14,17,2,3,3
6,71,19,67,0.22222,4,77,29,Pastrnak,1,72,8477956,4,100,1.29870,R,10,33,20252026,0,0,0.11111,R,261,David Pastrnak,BOS,1239.3376,149.7656,34.6180,398.6906,6.8550,41,5,102,9,57,10,0.435708,0.184081,0.380211,3,3,14,8,29,29,0,13,15,49,7,19,11,44,2,3,12
7,62,29,76,0.42857,5,78,38,Necas,0,30,8480039,47,100,1.28205,C,9,24,20252026,0,0,0.18446,R,206,Martin Necas,COL,1289.8846,150.2805,38.4498,466.3214,8.1924,25,2,84,14,54,12,0.454715,0.181042,0.364242,0,3,11,5,14,37,0,8,7,49,2,10,16,33,0,4,7
8,62,18,54,0.56935,3,65,35,Draisaitl,1,26,8477934,13,97,1.49230,C,16,42,20252026,1,1,0.18817,L,186,Leon Draisaitl,EDM,1294.5076,138.8220,37.4597,343.2309,6.7380,10,0,49,12,53,15,0.466764,0.173188,0.360048,1,2,5,5,21,5,1,5,0,48,3,7,26,23,0,29,5
9,51,30,55,0.33333,9,82,45,Robertson,1,32,8480027,22,96,1.17073,L,15,41,20252026,0,0,0.15306,L,294,Jason Robertson,DAL,1215.1829,145.1306,36.4681,413.6000,6.4472,31,4,98,11,108,23,0.455439,0.175916,0.368645,2,3,14,13,32,41,1,16,8,95,7,12,9,25,0,7,9


In [ ]:
'''
def labelWinners(year, first_ids, second_ids, third_ids, rank=False, edge=False):       #modified version of labelwinners for rr2
    
    purpose:    fetches a dataset of skaters and adds two columns: average TOI (extra feature) and either rrWinner or rrRank (target features) 
    parameters: -year (string) of a valid RR winner year; in yyyy or yyyyyyyy format
                -first_ids (list), a list of player_ids (strings) that won the award in question 
                -second_ids (list), a list of player_ids (strings) that were runner-ups of the award in question
                -third_ids (list), a list of player_ids (strings) that were third place finalists of the award in question
                -rank (boolean) if the dev wants labels to be one-hot encoded OR ranked by top 3 finalists
                -edge (boolean) if the dev is labeling winners on the combined EDGE set
    returns:    returns a dataframe of the selected year skaters with the labeled RR winner/finalists
    
    #format year for the filter (is a string when inputted)
    if len(year) == 4:
        int_year = int(year)
        interval = (int_year,int_year+1)
        year_interval = str(interval[0]) + str(interval[1])
    elif len(year) == 8:    #20252026
        year_interval = year
    else:
        raise SyntaxError("requires yyyy or yyyyyyyy format of season year")
    
    if edge == True:
        csv_path = f"../data/api/EDGEstats/skatersEDGE{year_interval}.csv"
        df = pd.read_csv(csv_path)
        regularDf = fetchSkaterStats(year=year, csv=False, edge=False)     #then combine it with regular GSS
        df = regularDf.merge(df)

    else:
        csv_path = f"../data/api/skaters/skaters{year_interval}.csv"
        df = pd.read_csv(csv_path)



    #add averageTOI
    df['averageTOI'] = np.zeros(df.shape[0])
    df['averageTOI'] = (df['timeOnIcePerGame']/60)
    startYear = year_interval[:4]               #get the first year in this season's year interval

    #add rr Winner ONLY
    if rank == False:       
        df['rrWinner'] = np.zeros(df.shape[0])      #create a new column for ranking
        for entry in first_ids:
            split = entry[0].rsplit("-")
            if str(split[0]) == (startYear):
                winner = entry[1]
                break
        df.loc[df['playerId'] == int(winner), 'rrWinner'] = 1   #modify the entry directly

    #add rr finalists
    else:
        df['rrRank'] = np.zeros(df.shape[0])      #create a new column for ranking
        for first in first_ids:
            season_year = first[0]
            season_year = season_year.rsplit("-")
            season_year = season_year[0]
            if season_year == startYear:
                winner = first[1]
                df.loc[df['playerId'] == int(winner), 'rrRank'] = 1
        
        for second in second_ids:
            season_year = second[0]
            season_year = season_year.rsplit("-")
            season_year = season_year[0]
            if season_year == startYear:
                runner_up = second[1]
                df.loc[df['playerId'] == int(runner_up), 'rrRank'] = 2
            
        for third in third_ids:
            season_year = third[0]
            season_year = season_year.rsplit("-")
            season_year = season_year[0]
            if season_year == startYear:
                finalist = third[1]
                df.loc[df['playerId'] == int(finalist), 'rrRank'] = 3
    return df
'''

In [56]:
#get the playerIds of the top 3
rr2firstids = placeToStats(rocket_richard[['szn','winner']])
rr2secondids = placeToStats(rocket_richard[['szn','runner_up']])
rr2thirdids = placeToStats(rocket_richard[['szn','finalist']])

''''''
if ranks == False:
    ska25 = labelWinners("2025",first_ids=rr2firstids,second_ids=rr2secondids,third_ids=rr2thirdids, rank=False, edge=True)
else:
    ska25 = labelWinners("2025",first_ids=rr2firstids,second_ids=rr2secondids,third_ids=rr2thirdids, rank=True, edge=True)


In [57]:
#split training sets and testing set
edgeSzns = ['20212022','20222023','20232024','20242025','20252026']
trainingSets = []
testingSet = []

to_drop = ['positionCode','lastName','teamAbbrevs','shGoals','shPoints']    #keep full name instead of last name
for year in edgeSzns:
    if ranks == True:
        modifiedDf = labelWinners(year=year[:4], first_ids=rr2firstids, second_ids=rr2secondids, third_ids=rr2thirdids, rank=True, edge=True)
    else:
        modifiedDf = labelWinners(year=year[:4], first_ids=rr2firstids, second_ids=rr2secondids, third_ids=rr2thirdids, rank=False, edge=True)
    
    feature_df = modifiedDf.drop(columns=to_drop)
    feature_df = feature_df.dropna()
    feature_df .loc[feature_df ['shootsCatches'] == "L", 'shootsCatches'] = 0    #encode L -> 0 to fit model
    feature_df .loc[feature_df ['shootsCatches'] == "R", 'shootsCatches'] = 1    #encode R -> 1 to fit model
    
    if year == "20252026":  #2025-2026 will be the testing year
        testingSet.append(feature_df)
    else:
        trainingSets.append(feature_df)
print(len(testingSet), len(trainingSets))

masterTraining = pd.DataFrame()
for train in trainingSets:
    masterTraining = pd.concat([masterTraining,train])

masterTesting = pd.DataFrame()
masterTesting = pd.concat([masterTesting,testingSet[0]])

1 4


In [58]:
#with pd.option_context("display.max_rows", None, "display.max_columns", None):     #view whole dataset
#        display(masterTraining)

### Milestone 3: Train model

In [59]:
if ranks == False:
    train_x = masterTraining.drop(columns=['skaterFullName','rrWinner','playerId','seasonId'])
    train_y = masterTraining['rrWinner']
else:
    train_x = masterTesting.drop(columns=['skaterFullName','rrRank','playerId','seasonId'])
    train_y = masterTesting['rrRank']

model = LogisticRegression(max_iter=20000, class_weight="balanced")
model.fit(train_x,train_y)

,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",20000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Defau

### Milestone 4: Test on EDGE + GSS of 2025-2026

In [60]:
test2025 = masterTesting
if ranks == False:
    test_x = test2025.drop(columns=['skaterFullName','rrWinner','playerId','seasonId'])
    test_y = test2025['rrWinner']
else:
    test_x = test2025.drop(columns=['skaterFullName','rrRank','playerId','seasonId'])
    test_y = test2025['rrRank']

pred_y = model.predict(test_x)
predictions = pd.Series(pred_y)
predictions = predictions.rename('predictions')
predictions = predictions.to_frame()
results = test2025.join(predictions)


In [61]:
if ranks == False:
    show = results.loc[results['predictions'] == 1]
else:
    show = results.loc[results['predictions'] != 0]
    show = show.dropna()
    hyman = results.loc[results['rrRank'] != 0]

#view influences on the model
feature_names = train_x.columns
coefficients = pd.Series(model.coef_[0],index=feature_names)
print(coefficients.sort_values())

with pd.option_context("display.max_rows", None, "display.max_columns", None):      #view full prediction list
    display(show)  

plusMinus                      -0.097154
L Net Side Shots               -0.069426
shots                          -0.069340
evGoals                        -0.058335
points                         -0.057893
goals                          -0.054619
totalDistanceSkated            -0.043672
evPoints                       -0.043271
midGoals                       -0.025403
Crease Shots                   -0.019812
gameWinningGoals               -0.019607
Outside R Shots                -0.018103
R Circle Shots                 -0.017941
highShots                      -0.016225
highGoals                      -0.016050
ppPoints                       -0.014959
Behind the Net Shots           -0.014774
R Net Side Shots               -0.013881
Outside L Shots                -0.013131
otGoals                        -0.012437
L Circle Shots                 -0.011309
L Corner Shots                 -0.008700
R Corner Shots                 -0.006611
Beyond Red Line Shots          -0.005331
assists         

,assists,evGoals,evPoints,faceoffWinPct,gameWinningGoals,gamesPlayed,goals,otGoals,penaltyMinutes,playerId,plusMinus,points,pointsPerGame,ppGoals,ppPoints,seasonId,shootingPct,shootsCatches,shots,skaterFullName,timeOnIcePerGame,topShotSpeed,skatingSpeed,totalDistanceSkated,distanceMaxGame,longShots,longGoals,midShots,midGoals,highShots,highGoals,offensiveZonePctg,neutralZonePctg,defensiveZonePctg,Behind the Net Shots,Beyond Red Line Shots,Center Point Shots,Crease Shots,High Slot Shots,L Circle Shots,L Corner Shots,L Net Side Shots,L Point Shots,Low Slot Shots,Offensive Neutral Zone Shots,Outside L Shots,Outside R Shots,R Circle Shots,R Corner Shots,R Net Side Shots,R Point Shots,averageTOI,rrRank,predictions
0,90,34,82,0.49512,4,82,48,1,44,8478402,17,138,1.68292,13,54,20252026,0.15686,0,306,Connor McDavid,1379.1219,132.0467,39.6089,531.4875,7.9080,8,1,92,9,120,26,0.476879,0.169191,0.353930,11,1,4,23,27,36,1,34,3,97,3,15,9,29,0,12,1,22.985365,3.0,3.0
2,74,42,97,0.51049,7,80,53,1,39,8477492,57,127,1.58750,11,30,20252026,0.15142,1,350,Nathan MacKinnon,1335.7125,146.4342,39.8786,456.0059,7.8713,32,6,149,20,83,19,0.466145,0.180970,0.352885,3,3,16,11,42,77,1,19,11,72,5,38,12,30,0,5,5,22.261875,1.0,1.0
14,50,28,56,0.47826,5,81,38,2,61,8477404,13,88,1.08641,8,30,20252026,0.17272,0,220,Jake Guentzel,1214.7530,133.6560,35.8189,405.7720,6.8168,10,0,62,8,116,26,0.439654,0.177980,0.382366,1,1,3,13,21,19,0,3,5,103,2,7,5,22,0,13,2,20.245883,0.0,2.0


findings/conclusions on the above can be found in commitlog.md